In [ ]:
# --- Step 1: Import libraries ---
import pandas as pd
import matplotlib.pyplot as plt

# --- Step 2: Load results ---
arima_results = pd.read_csv('../outputs/forecasts/arima_forecasts.csv')
xgb_results = pd.read_csv('../outputs/forecasts/xgboost_forecasts.csv')

# --- Step 3: Merge and compare ---
combined = pd.merge(arima_results, xgb_results, on='Industry', how='outer')

# --- Step 4: Compare model performance ---
combined['Best_Model'] = combined.apply(
    lambda x: 'XGBoost' if x['XGB_MAPE'] < x['ARIMA_MAPE'] else 'ARIMA', axis=1
)

# --- Step 5: Rank industries by growth ---
combined['Avg_Growth_%'] = (combined['ARIMA_Growth_%'] + combined['XGB_Growth_%']) / 2
growth_ranked = combined.sort_values('Avg_Growth_%', ascending=False)

# --- Step 6: Display top/bottom industries ---
print("Top 3 Growing Industries:")
display(growth_ranked.head(3))

print("\nTop 3 Declining Industries:")
display(growth_ranked.tail(3))

# --- Step 7: Visualization ---
plt.figure(figsize=(10,6))
plt.barh(growth_ranked['Industry'], growth_ranked['Avg_Growth_%'], color='skyblue')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Projected Industry Growth Rates (ARIMA + XGBoost Average)')
plt.xlabel('Forecasted Growth (%)')
plt.ylabel('Industry')
plt.tight_layout()
plt.show()

# --- Step 8: Save final combined results ---
growth_ranked.to_csv('../outputs/forecasts/final_combined_results.csv', index=False)
